# Experiment 6 - Zero-Shot System One Decision Model (`Jev`)

Evaluate TypeSafe AI's **Jev** (`jev-1.14.0`, a non-autoregressive System One model trained with Reinforcement Learning for Calibrated Decisions, RLCD) zero-shot on both the held-out **native Bengali** test split (`bn-test`, $n=490$) and the **English** test split (`en-test`, $n=356$).

| Condition | Primitive | Rubric | Test Splits |
|---|---|---|---|
| **6a** `jev_choice` | `Choice` (4 discrete clinical classes) | `unified` (DSM-5) | `bn-test` ($n=490$), `en-test` ($n=356$) |
| **6b** `jev_score` | `Score` (4-level ordered ordinal rubric, $E[Y] = \sum_{k=1}^4 k \cdot p_k$) | `unified` (DSM-5) | `bn-test` ($n=490$), `en-test` ($n=356$) |
| **6c** `naseem_dda` | `Choice` & `Score` | `naseem_dda` (Reddit DDA ablation) | `en-test` ($n=356$) |

Both primitives (`Choice` and `Score`) are queried in a **single forward pass** per post via `typesafe-sdk`.
Because Jev requires no local GPU training and produces no multi-gigabyte checkpoint weights, this notebook can be run either locally on CPU or in Colab.

In [ ]:
# Install dependencies (only needed if running on Colab).
!pip install -q "typesafe-sdk>=0.7.5" openpyxl pandas scikit-learn

In [ ]:
# Configure TYPESAFE_API_KEY from local .env, environment, or Colab Secrets.
import os
from src.jev_client import load_env_key

api_key = load_env_key("TYPESAFE_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("TYPESAFE_API_KEY")
        if api_key:
            os.environ["TYPESAFE_API_KEY"] = api_key
            print("TYPESAFE_API_KEY loaded from Colab Secrets")
    except Exception:
        pass

assert os.environ.get("TYPESAFE_API_KEY"), (
    "TYPESAFE_API_KEY not found. Set it in Repository/.env or Colab Secrets."
)
print("TYPESAFE_API_KEY configured: OK")

In [ ]:
# Prepare stratified 80/10/10 splits (seed=42) and run offline + live 2-row smoke check.
!python scripts/prepare_splits.py --lang bn --seed 42
!python scripts/prepare_splits.py --lang en --seed 42
!python scripts/smoke_test_jev.py --live

## Run Zero-Shot Evaluation on `bn-test` ($n=490$) and `en-test` ($n=356$)

`evaluate_jev.py` queries both `Choice` and `Score` in a single request per item, caches raw responses in `outputs/experiment6/`, and writes `eval_test.json` reports to `results/experiment6/`.

In [ ]:
!python evaluate_jev.py --lang bn --split test --rubric unified --concurrency 10 --bootstrap 1000 --seed 42
!python evaluate_jev.py --lang en --split test --rubric unified --concurrency 10 --bootstrap 1000 --seed 42
!python evaluate_jev.py --lang en --split test --rubric naseem_dda --concurrency 10 --bootstrap 1000 --seed 42

## Summary Table

In [ ]:
import json
from pathlib import Path

configs = [
    ("6a Choice (BN unified)", "results/experiment6/6a_jev_choice_bn/eval_test.json"),
    ("6b Score  (BN unified)", "results/experiment6/6b_jev_score_bn/eval_test.json"),
    ("6a Choice (EN unified)", "results/experiment6/6a_jev_choice_en/eval_test.json"),
    ("6b Score  (EN unified)", "results/experiment6/6b_jev_score_en/eval_test.json"),
    ("6c Choice (EN naseem_dda)", "results/experiment6/6c_jev_choice_naseem_dda_en/eval_test.json"),
    ("6c Score  (EN naseem_dda)", "results/experiment6/6c_jev_score_naseem_dda_en/eval_test.json"),
]

print(f"{'Configuration':26s} | {'macro-F1':8s} | {'QWK':7s} | {'MAE':7s} | {'Acc':7s} | {'Down%':7s} | {'Up%':7s} | {'FScore':7s} | {'min/mild/mod/sev F1'}")
print("-" * 120)
for label, rel_path in configs:
    p = Path(rel_path)
    if not p.exists():
        continue
    r = json.loads(p.read_text(encoding="utf-8"))
    d = r["direction"]
    g = r["graded"]
    cls_f1 = " / ".join(f"{r[c]:.4f}" for c in ("minimal", "mild", "moderate", "severe"))
    print(
        f"{label:26s} | {r['macro_f1']:8.4f} | {r['qwk']:7.4f} | {r['mae']:7.4f} | "
        f"{r['accuracy']:7.4f} | {d['downgrade_rate']:7.4f} | {d['upgrade_rate']:7.4f} | "
        f"{g['graded_fscore']:7.4f} | {cls_f1}"
    )